In [ ]:
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
import numpy as np
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from rich.progress import track
from IPython.display import display

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
])

In [ ]:
train_data = ImageFolder(root='./train', transform=transform)
test_data = ImageFolder(root='./test', transform=transform)

In [ ]:
train_labels = train_data.targets
test_labels = test_data.targets

In [ ]:
train_loader = DataLoader(train_data, batch_size=10, shuffle=True)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)

In [ ]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv = nn.Sequential(
      nn.Conv2d(1,32,kernel_size=3,padding=1,stride=1),
      nn.ReLU(),
      nn.MaxPool2d(2,2),
      nn.Conv2d(32,64,kernel_size=3,padding=1,stride=1),
      nn.ReLU(),
      nn.MaxPool2d(2,2),
      nn.Conv2d(64,128,kernel_size=3,padding=1,stride=1),
      nn.ReLU(),
      nn.MaxPool2d(2,2)
    )

    self.fn = nn.Sequential(
      nn.Linear(128 * 3 * 3,10),
      nn.ReLU(),
      nn.Linear(10,10)
    )

  def forward(self,x):
    x = self.conv(x)
    x = torch.flatten(x,1)
    x = self.fn(x)
    return x

In [ ]:
test = torch.tensor([0])
for image,label in test_loader:
  for data in image:
    test = data
  # print(test)
  break

model = CNN()

output = None

with torch.no_grad():
  output = model.forward(test.unsqueeze(0))

output

model.forward(test.unsqueeze(0))

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [ ]:
for epoch in track(range(1000), description="Training Epoch"):
  for data in train_loader:
    images,labels = data
    y_pred = model.forward(images)
    y_true = labels
    loss = loss_fn(y_pred,y_true)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
index = 99
data = test_loader.dataset[index][0]
label = test_loader.dataset[index][1]

output = model.forward(data.unsqueeze(0))

In [ ]:
import torch.nn.functional as F

probabilities = F.softmax(output, dim=1)
display(probabilities)

predicted_class = torch.argmax(probabilities, dim=1)
display(predicted_class.item)

In [ ]:
correct_predictions = 0
total_predictions = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

accuracy = 100 * correct_predictions / total_predictions
print(f'Accuracy of the model on the {total_predictions} test images: {accuracy:.2f}%')